# RepoScout: Process Repository README Embeddings

This self-contained Databricks notebook reads RepoScout README data from Lakebase with Spark JDBC, performs deterministic Spark cleaning and chunking, generates normalized SentenceTransformer embeddings, validates the result, and transactionally persists pgvector rows with psycopg.

Run the Alembic migration before executing this notebook. The notebook does not import or depend on the FastAPI application.

## Install notebook-scoped dependencies

Versions are pinned so the processing configuration is reproducible. The Databricks Runtime supplies Spark, pandas, PyArrow, and the PostgreSQL JDBC connector.

In [ ]:
%pip install -q "databricks-sdk==0.121.0" "sentence-transformers==5.6.1" "psycopg[binary]==3.3.4"

In [ ]:
dbutils.library.restartPython()

## Configuration

In [ ]:
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from typing import Iterator

import pandas as pd
import psycopg
import sentence_transformers
from databricks.sdk import WorkspaceClient
from pyspark import StorageLevel
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType


def ensure_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


ensure_widget("lakebase_endpoint", os.getenv("LAKEBASE_ENDPOINT", ""), "Lakebase endpoint resource name")
ensure_widget("pg_host", os.getenv("PGHOST", ""), "PostgreSQL host")
ensure_widget("pg_port", os.getenv("PGPORT", "5432"), "PostgreSQL port")
ensure_widget("pg_database", os.getenv("PGDATABASE", ""), "PostgreSQL database")
ensure_widget("pg_user", os.getenv("PGUSER", ""), "PostgreSQL OAuth role")
ensure_widget("pg_sslmode", os.getenv("PGSSLMODE", "require"), "PostgreSQL SSL mode")
ensure_widget("max_repositories", "5", "Maximum changed repositories to process")
ensure_widget("chunk_size", "800", "Chunk size in characters")
ensure_widget("chunk_overlap", "100", "Chunk overlap in characters")
ensure_widget("embedding_batch_size", "32", "SentenceTransformer batch size")
ensure_widget("embedding_partitions", "1", "Spark embedding partitions")


def required_widget(name: str) -> str:
    value = dbutils.widgets.get(name).strip()
    if not value:
        raise ValueError(f"Widget {name!r} is required")
    return value


LAKEBASE_ENDPOINT = required_widget("lakebase_endpoint")
PGHOST = required_widget("pg_host")
PGPORT = int(required_widget("pg_port"))
PGDATABASE = required_widget("pg_database")
PGUSER = required_widget("pg_user")
PGSSLMODE = required_widget("pg_sslmode")
MAX_REPOSITORIES = int(required_widget("max_repositories"))
CHUNK_SIZE = int(required_widget("chunk_size"))
CHUNK_OVERLAP = int(required_widget("chunk_overlap"))
EMBEDDING_BATCH_SIZE = int(required_widget("embedding_batch_size"))
EMBEDDING_PARTITIONS = int(required_widget("embedding_partitions"))

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
NORMALIZE_EMBEDDINGS = True
CLEANING_VERSION = "reposcout-markdown-v1"

if not 1 <= PGPORT <= 65535:
    raise ValueError("pg_port must be between 1 and 65535")
if MAX_REPOSITORIES < 1:
    raise ValueError("max_repositories must be positive")
if CHUNK_SIZE < 1 or CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("chunk_size must be positive and chunk_overlap must be in [0, chunk_size)")
if EMBEDDING_BATCH_SIZE < 1 or EMBEDDING_PARTITIONS < 1:
    raise ValueError("embedding_batch_size and embedding_partitions must be positive")

processing_config = {
    "cleaning_version": CLEANING_VERSION,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": EMBEDDING_DIM,
    "normalize_embeddings": NORMALIZE_EMBEDDINGS,
    "sentence_transformers_version": sentence_transformers.__version__,
}
PROCESSING_CONFIG_HASH = hashlib.sha256(
    json.dumps(processing_config, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
RUN_STARTED_AT = datetime.now(timezone.utc)
RUN_STARTED_MONOTONIC = time.monotonic()

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Processing configuration: {PROCESSING_CONFIG_HASH}")
print(f"Repository limit: {MAX_REPOSITORIES}")

## Independent Lakebase OAuth and Spark JDBC reads

A new short-lived credential is generated immediately before each eager JDBC read. Credentials are never placed in widgets, URLs, output, or application settings.

In [ ]:
workspace = WorkspaceClient()
JDBC_URL = f"jdbc:postgresql://{PGHOST}:{PGPORT}/{PGDATABASE}?sslmode={PGSSLMODE}"


def generate_database_credential() -> str:
    try:
        credential = workspace.postgres.generate_database_credential(
            endpoint=LAKEBASE_ENDPOINT
        )
    except Exception:
        raise RuntimeError("Unable to generate a Lakebase database credential") from None
    if not credential.token:
        raise RuntimeError("Lakebase returned an empty database credential")
    return credential.token


def read_lakebase_with_spark(query: str) -> DataFrame:
    credential = generate_database_credential()
    try:
        frame = (
            spark.read.format("jdbc")
            .option("url", JDBC_URL)
            .option("driver", "org.postgresql.Driver")
            .option("query", query)
            .option("user", PGUSER)
            .option("password", credential)
            .option("fetchsize", "500")
            .option("numPartitions", "1")
            .load()
        )
        return frame.localCheckpoint(eager=True)
    except Exception:
        raise RuntimeError("Unable to read Lakebase through Spark JDBC") from None
    finally:
        credential = None


repositories_df = read_lakebase_with_spark(
    """
    SELECT repo_id, full_name, description, primary_language, html_url, stars
    FROM public.repositories
    """
)
readmes_df = read_lakebase_with_spark(
    """
    SELECT repo_id, raw_content, lower(content_hash) AS content_hash,
           retrieval_status, retrieved_at
    FROM public.repository_readmes
    """
)
chunk_state_df = read_lakebase_with_spark(
    """
    SELECT DISTINCT repo_id, lower(source_content_hash) AS source_content_hash,
           embedding_model, lower(processing_config_hash) AS processing_config_hash
    FROM public.repository_chunks
    """
)

source_df = repositories_df.join(readmes_df, "repo_id", "inner").localCheckpoint(eager=True)
SOURCE_README_COUNT = source_df.count()
print(f"Source README rows: {SOURCE_README_COUNT}")

## Eligibility and incremental filtering

In [ ]:
available_df = source_df.filter(
    (F.col("retrieval_status") == F.lit("available"))
    & F.col("raw_content").isNotNull()
    & (F.length(F.trim(F.col("raw_content"))) > 0)
    & F.col("content_hash").rlike("^[0-9a-f]{64}$")
)

eligible_df = (
    available_df
    .withColumn("embedding_model", F.lit(EMBEDDING_MODEL))
    .withColumn("processing_config_hash", F.lit(PROCESSING_CONFIG_HASH))
    .localCheckpoint(eager=True)
)

state_keys_df = chunk_state_df.select(
    "repo_id", "source_content_hash", "embedding_model", "processing_config_hash"
)
eligible_keys = ["repo_id", "source_content_hash", "embedding_model", "processing_config_hash"]
eligible_with_source_hash_df = eligible_df.withColumnRenamed(
    "content_hash", "source_content_hash"
)

unchanged_df = eligible_with_source_hash_df.join(
    state_keys_df, eligible_keys, "left_semi"
)
changed_df = eligible_with_source_hash_df.join(
    state_keys_df, eligible_keys, "left_anti"
).localCheckpoint(eager=True)
selected_df = changed_df.orderBy("repo_id").limit(MAX_REPOSITORIES).localCheckpoint(eager=True)

existing_chunk_repos_df = chunk_state_df.select("repo_id").distinct()
missing_with_chunks_df = (
    readmes_df.filter(F.col("retrieval_status") == F.lit("missing"))
    .select("repo_id")
    .join(existing_chunk_repos_df, "repo_id", "inner")
    .distinct()
    .localCheckpoint(eager=True)
)

ELIGIBLE_REPOSITORY_COUNT = eligible_df.count()
UNCHANGED_REPOSITORY_COUNT = unchanged_df.count()
CHANGED_REPOSITORY_COUNT = changed_df.count()
SELECTED_REPOSITORY_COUNT = selected_df.count()
MISSING_WITH_CHUNKS_COUNT = missing_with_chunks_df.count()

print(f"Eligible repositories: {ELIGIBLE_REPOSITORY_COUNT}")
print(f"Unchanged repositories: {UNCHANGED_REPOSITORY_COUNT}")
print(f"Changed/new repositories: {CHANGED_REPOSITORY_COUNT}")
print(f"Selected repositories: {SELECTED_REPOSITORY_COUNT}")
print(f"Missing repositories with stale chunks: {MISSING_WITH_CHUNKS_COUNT}")

## Spark Markdown cleaning and deterministic chunking

In [ ]:
def clean_readmes(frame: DataFrame) -> DataFrame:
    text = F.col("raw_content")
    text = F.regexp_replace(text, r"\r\n?", "\n")
    text = F.regexp_replace(text, r"(?s)<!--.*?-->", " ")
    text = F.regexp_replace(text, r"!\[([^]]*)\]\([^)]+\)", "$1")
    text = F.regexp_replace(text, r"\[([^]]+)\]\([^)]+\)", "$1")
    text = F.regexp_replace(text, r"<(https?://[^>]+)>", "$1")
    text = F.regexp_replace(text, r"(?s)<[^>]+>", " ")
    text = F.regexp_replace(text, r"(?m)^\s*(```|~~~)[^\n]*$", "")
    text = F.regexp_replace(text, r"(?m)^\s{0,3}#{1,6}\s*", "")
    text = F.regexp_replace(text, r"(?m)^\s*>+\s?", "")
    text = F.regexp_replace(text, r"(?m)^\s*(?:[-+*]|\d+[.)])\s+", "")
    text = F.regexp_replace(text, r"(?m)^\s*([-*_])(?:\s*\1){2,}\s*$", "")
    text = F.regexp_replace(text, r"`+", "")
    text = F.regexp_replace(text, r"&nbsp;", " ")
    text = F.regexp_replace(text, r"&amp;", "&")
    text = F.regexp_replace(text, r"[\t\x0B\f ]+", " ")
    text = F.regexp_replace(text, r" *\n *", "\n")
    text = F.regexp_replace(text, r"\n{3,}", "\n\n")
    return frame.withColumn("cleaned_text", F.trim(text))


def chunk_readmes(frame: DataFrame, chunk_size: int, chunk_overlap: int) -> DataFrame:
    step = chunk_size - chunk_overlap
    with_counts = (
        frame
        .withColumn("text_length", F.length("cleaned_text"))
        .withColumn(
            "chunk_count",
            (
                F.ceil(
                    F.greatest(F.col("text_length") - F.lit(chunk_size), F.lit(0))
                    / F.lit(step)
                )
                + F.lit(1)
            ).cast("int"),
        )
    )
    chunks = (
        with_counts
        .withColumn("chunk_index", F.explode(F.sequence(F.lit(0), F.col("chunk_count") - 1)))
        .withColumn("chunk_start", F.col("chunk_index") * F.lit(step) + F.lit(1))
        .withColumn(
            "chunk_text",
            F.trim(F.expr(f"substring(cleaned_text, chunk_start, {chunk_size})")),
        )
        .filter(F.length("chunk_text") > 0)
        .withColumn(
            "chunk_id",
            F.sha2(
                F.concat_ws(
                    ":",
                    F.col("repo_id").cast("string"),
                    F.col("source_content_hash"),
                    F.col("processing_config_hash"),
                    F.col("chunk_index").cast("string"),
                ),
                256,
            ),
        )
    )
    return chunks.select(
        "chunk_id", "repo_id", "full_name", "chunk_index", "chunk_text",
        "source_content_hash", "embedding_model", "processing_config_hash"
    )


# Notebook-native deterministic transformation checks.
_cleaning_fixture = spark.createDataFrame(
    [(1, "# Title\r\n\r\n[Docs](https://example.com)\n\n```python\nprint('ok')\n```")],
    ["repo_id", "raw_content"],
)
_cleaned_fixture = clean_readmes(_cleaning_fixture).select("cleaned_text").first()[0]
assert _cleaned_fixture == "Title\n\nDocs\n\nprint('ok')", _cleaned_fixture

_chunk_fixture = spark.createDataFrame(
    [(1, "fixture/repo", "abcdefghijkl", "a" * 64, EMBEDDING_MODEL, PROCESSING_CONFIG_HASH)],
    ["repo_id", "full_name", "cleaned_text", "source_content_hash", "embedding_model", "processing_config_hash"],
)
_chunk_fixture_rows = chunk_readmes(_chunk_fixture, 5, 2).orderBy("chunk_index").collect()
assert [(row.chunk_index, row.chunk_text) for row in _chunk_fixture_rows] == [
    (0, "abcde"), (1, "defgh"), (2, "ghijk"), (3, "jkl")
]

cleaned_selected_df = clean_readmes(selected_df).localCheckpoint(eager=True)
usable_selected_df = cleaned_selected_df.filter(F.length("cleaned_text") > 0)
chunks_df = chunk_readmes(usable_selected_df, CHUNK_SIZE, CHUNK_OVERLAP).localCheckpoint(eager=True)

USABLE_REPOSITORY_COUNT = usable_selected_df.select("repo_id").distinct().count()
UNUSABLE_REPOSITORY_COUNT = SELECTED_REPOSITORY_COUNT - USABLE_REPOSITORY_COUNT
GENERATED_CHUNK_COUNT = chunks_df.count()
print(f"Usable selected repositories: {USABLE_REPOSITORY_COUNT}")
print(f"Unusable selected repositories: {UNUSABLE_REPOSITORY_COUNT}")
print(f"Generated chunks: {GENERATED_CHUNK_COUNT}")

## Batched SentenceTransformer embeddings

In [ ]:
_worker_model = None


@pandas_udf(ArrayType(FloatType()))
def embed_text_batches(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    global _worker_model
    if _worker_model is None:
        from sentence_transformers import SentenceTransformer

        _worker_model = SentenceTransformer(
            EMBEDDING_MODEL,
            cache_folder="/tmp/reposcout-huggingface",
        )
    for texts in iterator:
        vectors = _worker_model.encode(
            texts.astype(str).tolist(),
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        yield pd.Series([vector.astype("float32").tolist() for vector in vectors])


def add_embeddings(frame: DataFrame) -> DataFrame:
    return frame.repartition(EMBEDDING_PARTITIONS, "repo_id").withColumn(
        "embedding", embed_text_batches("chunk_text")
    )


# Exercise actual worker-side inference when this run has source work.
# An unchanged rerun avoids loading the model entirely.
if SELECTED_REPOSITORY_COUNT:
    _embedding_fixture = spark.createDataFrame(
        [(1, "FastAPI is a modern Python web framework."), (2, "PostgreSQL stores vectors.")],
        ["id", "chunk_text"],
    )
    _embedding_fixture_rows = add_embeddings(_embedding_fixture).collect()
    assert all(len(row.embedding) == EMBEDDING_DIM for row in _embedding_fixture_rows)
    assert all(
        abs(math.sqrt(sum(float(value) ** 2 for value in row.embedding)) - 1.0) <= 1e-3
        for row in _embedding_fixture_rows
    )

PROCESSED_AT = datetime.now(timezone.utc)
embedded_df = (
    add_embeddings(chunks_df)
    .withColumn("processed_at", F.lit(PROCESSED_AT))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
EMBEDDED_CHUNK_COUNT = embedded_df.count()
print(f"Embedded chunks: {EMBEDDED_CHUNK_COUNT}")

## Spark validation

In [ ]:
def assert_zero(count: int, message: str) -> None:
    if count:
        raise AssertionError(f"{message}: {count}")


assert_zero(
    embedded_df.groupBy("chunk_id").count().filter(F.col("count") != 1).count(),
    "Duplicate chunk IDs",
)
assert_zero(
    embedded_df.groupBy("repo_id", "chunk_index").count().filter(F.col("count") != 1).count(),
    "Duplicate repository chunk indexes",
)
assert_zero(
    embedded_df.filter(
        (F.length("chunk_text") == 0) | (F.length("chunk_text") > CHUNK_SIZE)
    ).count(),
    "Invalid chunk lengths",
)
assert_zero(
    embedded_df.filter(
        F.col("embedding").isNull() | (F.size("embedding") != EMBEDDING_DIM)
    ).count(),
    "Invalid embedding dimensions",
)
assert_zero(
    embedded_df.filter(F.expr("exists(embedding, x -> x IS NULL OR isnan(x))")).count(),
    "Non-finite embeddings",
)
assert_zero(
    embedded_df.withColumn(
        "embedding_norm",
        F.expr(
            "sqrt(aggregate(embedding, cast(0.0 as double), "
            "(acc, x) -> acc + cast(x as double) * cast(x as double)))"
        ),
    ).filter(F.abs(F.col("embedding_norm") - F.lit(1.0)) > F.lit(1e-3)).count(),
    "Non-normalized embeddings",
)
assert_zero(
    embedded_df.groupBy("repo_id").agg(
        F.min("chunk_index").alias("minimum"),
        F.max("chunk_index").alias("maximum"),
        F.count("*").alias("row_count"),
        F.countDistinct("chunk_index").alias("distinct_count"),
    ).filter(
        (F.col("minimum") != 0)
        | (F.col("maximum") != F.col("row_count") - 1)
        | (F.col("distinct_count") != F.col("row_count"))
    ).count(),
    "Non-contiguous chunk indexes",
)
assert_zero(
    embedded_df.select("repo_id", "source_content_hash").distinct().join(
        selected_df.select(
            "repo_id", F.col("source_content_hash").alias("selected_source_hash")
        ),
        "repo_id",
        "left",
    ).filter(
        F.col("selected_source_hash").isNull()
        | (F.col("source_content_hash") != F.col("selected_source_hash"))
    ).count(),
    "Chunk source mapping errors",
)
if EMBEDDED_CHUNK_COUNT != GENERATED_CHUNK_COUNT:
    raise AssertionError("Embedding output count does not match generated chunk count")

print(f"Validated {EMBEDDED_CHUNK_COUNT} embeddings with dimension {EMBEDDING_DIM}")

## Transactional pgvector persistence

Spark has completed and validated the processing pipeline. Psycopg is used only for the final atomic replacement because direct Spark serialization of PostgreSQL vector values is unnecessarily fragile.

In [ ]:
INSERT_CHUNK_SQL = """
    INSERT INTO public.repository_chunks (
        chunk_id, repo_id, chunk_index, chunk_text, source_content_hash,
        embedding, embedding_model, processing_config_hash, processed_at
    ) VALUES (%s, %s, %s, %s, %s, %s::vector, %s, %s, %s)
    ON CONFLICT (repo_id, chunk_index) DO UPDATE SET
        chunk_id = EXCLUDED.chunk_id,
        chunk_text = EXCLUDED.chunk_text,
        source_content_hash = EXCLUDED.source_content_hash,
        embedding = EXCLUDED.embedding,
        embedding_model = EXCLUDED.embedding_model,
        processing_config_hash = EXCLUDED.processing_config_hash,
        processed_at = EXCLUDED.processed_at
"""


def vector_literal(values: list[float]) -> str:
    return "[" + ",".join(format(float(value), ".9g") for value in values) + "]"


def connect_lakebase() -> psycopg.Connection:
    credential = generate_database_credential()
    try:
        return psycopg.connect(
            host=PGHOST,
            port=PGPORT,
            dbname=PGDATABASE,
            user=PGUSER,
            password=credential,
            sslmode=PGSSLMODE,
            connect_timeout=30,
        )
    except Exception:
        raise RuntimeError("Unable to connect to Lakebase for persistence") from None
    finally:
        credential = None


selected_sources = {
    int(row.repo_id): row.source_content_hash
    for row in selected_df.select("repo_id", "source_content_hash").collect()
}
missing_repo_ids = [
    int(row.repo_id) for row in missing_with_chunks_df.select("repo_id").collect()
]

HASH_RACE_REPOSITORY_COUNT = 0
REPLACED_REPOSITORY_COUNT = 0
DELETED_STALE_REPOSITORY_COUNT = 0
PERSISTED_THIS_RUN_COUNT = 0
PERSISTED_CHUNK_COUNT = 0
HNSW_INDEX_EXISTS = False

try:
    with connect_lakebase() as connection:
        with connection.transaction():
            current_sources: dict[int, str] = {}
            if selected_sources:
                cursor = connection.execute(
                    """
                    SELECT repo_id, lower(content_hash)
                    FROM public.repository_readmes
                    WHERE repo_id = ANY(%s)
                      AND retrieval_status = 'available'
                    FOR SHARE
                    """,
                    (list(selected_sources),),
                )
                current_sources = {
                    int(repo_id): content_hash
                    for repo_id, content_hash in cursor.fetchall()
                    if content_hash is not None
                }

            valid_repo_ids = {
                repo_id
                for repo_id, source_hash in selected_sources.items()
                if current_sources.get(repo_id) == source_hash
            }
            HASH_RACE_REPOSITORY_COUNT = len(selected_sources) - len(valid_repo_ids)

            current_missing_repo_ids: set[int] = set()
            if missing_repo_ids:
                cursor = connection.execute(
                    """
                    SELECT repo_id
                    FROM public.repository_readmes
                    WHERE repo_id = ANY(%s)
                      AND retrieval_status = 'missing'
                    FOR SHARE
                    """,
                    (missing_repo_ids,),
                )
                current_missing_repo_ids = {int(row[0]) for row in cursor.fetchall()}

            delete_repo_ids = sorted(valid_repo_ids | current_missing_repo_ids)
            if delete_repo_ids:
                connection.execute(
                    "DELETE FROM public.repository_chunks WHERE repo_id = ANY(%s)",
                    (delete_repo_ids,),
                )

            batch: list[tuple[object, ...]] = []
            with connection.cursor() as cursor:
                for row in embedded_df.toLocalIterator():
                    if int(row.repo_id) not in valid_repo_ids:
                        continue
                    batch.append(
                        (
                            row.chunk_id,
                            int(row.repo_id),
                            int(row.chunk_index),
                            row.chunk_text,
                            row.source_content_hash,
                            vector_literal(row.embedding),
                            row.embedding_model,
                            row.processing_config_hash,
                            row.processed_at,
                        )
                    )
                    if len(batch) >= 100:
                        cursor.executemany(INSERT_CHUNK_SQL, batch)
                        PERSISTED_THIS_RUN_COUNT += len(batch)
                        batch.clear()
                if batch:
                    cursor.executemany(INSERT_CHUNK_SQL, batch)
                    PERSISTED_THIS_RUN_COUNT += len(batch)

            REPLACED_REPOSITORY_COUNT = len(valid_repo_ids)
            DELETED_STALE_REPOSITORY_COUNT = len(current_missing_repo_ids)

        PERSISTED_CHUNK_COUNT = connection.execute(
            "SELECT COUNT(*) FROM public.repository_chunks"
        ).fetchone()[0]
        index_row = connection.execute(
            """
            SELECT indexdef
            FROM pg_indexes
            WHERE schemaname = 'public'
              AND tablename = 'repository_chunks'
              AND indexname = 'ix_repository_chunks_embedding_hnsw'
            """
        ).fetchone()
        HNSW_INDEX_EXISTS = bool(index_row and "USING hnsw" in index_row[0])
except Exception:
    raise RuntimeError("Unable to transactionally persist repository chunks") from None
finally:
    embedded_df.unpersist()

if not HNSW_INDEX_EXISTS:
    raise AssertionError("Expected repository chunk HNSW index is missing")

print(f"Repositories replaced: {REPLACED_REPOSITORY_COUNT}")
print(f"Stale missing repositories cleared: {DELETED_STALE_REPOSITORY_COUNT}")
print(f"Chunks persisted this run: {PERSISTED_THIS_RUN_COUNT}")
print(f"Total persisted chunks: {PERSISTED_CHUNK_COUNT}")
print("HNSW index confirmed")

## Run summary

In [ ]:
summary = {
    "source_readme_count": SOURCE_README_COUNT,
    "eligible_repository_count": ELIGIBLE_REPOSITORY_COUNT,
    "unchanged_repository_count": UNCHANGED_REPOSITORY_COUNT,
    "changed_repository_count": CHANGED_REPOSITORY_COUNT,
    "selected_repository_count": SELECTED_REPOSITORY_COUNT,
    "processed_repository_count": REPLACED_REPOSITORY_COUNT,
    "usable_repository_count": USABLE_REPOSITORY_COUNT,
    "unusable_repository_count": UNUSABLE_REPOSITORY_COUNT,
    "hash_race_repository_count": HASH_RACE_REPOSITORY_COUNT,
    "generated_chunk_count": GENERATED_CHUNK_COUNT,
    "validated_embedding_count": EMBEDDED_CHUNK_COUNT,
    "embedding_dimension": EMBEDDING_DIM,
    "replaced_repository_count": REPLACED_REPOSITORY_COUNT,
    "deleted_stale_repository_count": DELETED_STALE_REPOSITORY_COUNT,
    "persisted_this_run_count": PERSISTED_THIS_RUN_COUNT,
    "persisted_chunk_count": int(PERSISTED_CHUNK_COUNT),
    "hnsw_index_exists": HNSW_INDEX_EXISTS,
    "embedding_model": EMBEDDING_MODEL,
    "processing_config_hash": PROCESSING_CONFIG_HASH,
    "duration_seconds": round(time.monotonic() - RUN_STARTED_MONOTONIC, 3),
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
print(json.dumps(summary, indent=2, sort_keys=True))
display(spark.createDataFrame([summary]))